In [ ]:


import pandas as pd
import plotly.express as px
import os

# I am reading the Coffee Shop Sales Excel file directly
df = pd.read_excel("Coffee Shop Sales.xlsx")

# Cleaning the column names
df.columns = [c.strip() for c in df.columns]

# Showing first few rows
df.head()


# I am searching for a column containing 'date'
date_col = None
for c in df.columns:
    if "date" in c.lower():
        date_col = c
        break

# Converting date column into proper datetime
df["order_date"] = pd.to_datetime(df[date_col], errors="coerce")

# Creating month column for monthly sales trend
df["month"] = df["order_date"].dt.to_period("M").dt.to_timestamp()



# 2) Handling the Sales Column


# Trying to detect sales column automatically
sales_col = None
for c in df.columns:
    if any(keyword in c.lower() for keyword in ["sales", "total", "amount", "price", "revenue", "cost"]):
        sales_col = c
        break

if sales_col is None:
    print("Warning: Could not automatically detect sales column. Please check the available columns.")
    print(f"Available columns: {df.columns.tolist()}")
    # Fallback or ask user for input
    # For now, if sales_col is still None, the next line will likely fail with a KeyError,
    # but the columns will be printed for inspection.

# Creating a clean sales_value column
df["sales_value"] = pd.to_numeric(df[sales_col], errors="coerce")



# Creating Output Folder


OUT_DIR = "reports"
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(fig, name):
    fig.write_html(f"{OUT_DIR}/{name}.html")
    print("Saved:", f"{OUT_DIR}/{name}.html")



# VISUALIZATION 1 – Monthly Sales Trend


monthly = df.groupby("month")["sales_value"].sum().reset_index()

fig1 = px.line(
    monthly,
    x="month",
    y="sales_value",
    title="Monthly Sales Trend",
    markers=True
)
save_fig(fig1, "monthly_sales_trend")
fig1.show()


# VISUALIZATION 2 – Sales by Category



# finding category column
cat_col = None
for c in df.columns:
    if "category" in c.lower():
        cat_col = c
        break

cat = df.groupby(cat_col)["sales_value"].sum().reset_index()

fig2 = px.bar(
    cat,
    x=cat_col,
    y="sales_value",
    title="Sales by Category",
    text="sales_value"
)
fig2.update_traces(textposition="outside")
save_fig(fig2, "sales_by_category")
fig2.show()



# VISUALIZATION 3 – Top 15 Products

prod_col = None
for c in df.columns:
    if "product" in c.lower():
        prod_col = c
        break

top = (
    df.groupby(prod_col)["sales_value"].sum()
    .reset_index()
    .sort_values("sales_value", ascending=False)
    .head(15)
)

fig3 = px.bar(
    top,
    x=prod_col,
    y="sales_value",
    title="Top 15 Best-Selling Products"
)
fig3.update_layout(xaxis_tickangle=-45)
save_fig(fig3, "top_15_products")
fig3.show()



# VISUALIZATION 4 – Sales by Store Location


loc_col = None
for c in df.columns:
    if "location" in c.lower() or "store" in c.lower():
        loc_col = c
        break

loc = df.groupby(loc_col)["sales_value"].sum().reset_index()

fig4 = px.bar(
    loc,
    x=loc_col,
    y="sales_value",
    title="Sales by Store Location"
)
fig4.update_layout(xaxis_tickangle=-45)
save_fig(fig4, "sales_by_location")
fig4.show()


print("\n🎉 All 4 Visualizations Generated Successfully!")
print("📁 Saved in /reports folder")

Saved: reports/monthly_sales_trend.html


Saved: reports/sales_by_category.html


Saved: reports/top_15_products.html


Saved: reports/sales_by_location.html



🎉 All 4 Visualizations Generated Successfully!
📁 Saved in /reports folder
